In [64]:
import pandas as pd
import numpy as np
import pickle

In [65]:
df = pd.read_csv("../data/processed/clean_data.csv")
df.head()

,title,tags_x,genre,popularity,vote_count,vote_average
0,my little pony: a new generation,"robert cullen, josé lui ucha vanessa hudgens, ...","Animation, Family, Fantasy, Comedy, Music, Adv...",137.815,190.0,popular
1,the starling,"theodor melfi melissa mccarthy, chri o'dowd, k...",Drama,27.515,174.0,popular
2,je suis karl,"christian schwochow luna wedler, janni niewöhn...",Drama,19.294,20.0,not_popular
3,confessions of an invisible girl,"bruno garotti klara castanho, lucca picon, júl...","Comedy, Drama",62.097,144.0,average
4,intrusion,"adam salki freida pinto, logan marshall-green,...",Thriller,60.475,412.0,below_avg


In [66]:
print(df.shape)
print(df.columns)

(1726, 6)
Index(['title', 'tags_x', 'genre', 'popularity', 'vote_count', 'vote_average'], dtype='str')


# load content similarity matrix

In [67]:
similarity = pickle.load(
    open("../models/user_similarity.pkl",'rb')
)

In [68]:
similarity.shape

(1000, 1000)

# load collabrative similarity

In [69]:
user_similarity = pickle.load(
    open("../models/user_similarity.pkl",'rb')
)

In [70]:
user_similarity.shape

(1000, 1000)

# create movie collabrative score matrix

In [71]:
np.random.seed(42)

collab_scores = np.random.rand(
    len(df),
    len(df)
)

In [72]:
collab_scores.shape

(1726, 1726)

# normalize scores

In [73]:
content_scores= similarity/similarity.max()
collab_scores = collab_scores/collab_scores.max()

# weighted hybrid formula

In [74]:
ALPHA = .6
BETA = .4

# Create hybrid matrix

In [75]:
print("Content:", content_scores.shape)
print("Collaborative:", collab_scores.shape)
print("Movies:", len(df))

Content: (1000, 1000)
Collaborative: (1726, 1726)
Movies: 1726


In [76]:
similarity = pickle.load(
    open("../models/similarity.pkl","rb")
)

print(similarity.shape)

(1726, 1726)


In [77]:
collab_scores.shape

(1726, 1726)

In [78]:
hybrid_scores = (
    ALPHA * similarity +
    BETA * collab_scores
)

In [79]:
hybrid_scores.shape

(1726, 1726)

# recommendation function

In [80]:
def hybrid_recommend(movie,n=5):
    
    try:
        movie= movie.lower()
        titles = df['title'].str.lower()
   
        idx = df[
           titles == movie
        ].index[0]

    except:
        return"Movie not found"
    
    distances = list(
        enumerate(
            hybrid_scores[idx]
        )
    )

    movie_list = sorted(
        distances,
        reverse = True,
        key = lambda x:x[1]
    )[1:n+1]

    recommendations = []
    for i in movie_list:
        recommendations.append(
            df.iloc[1[0]].title
        )

    return recommendations    

<>:29: SyntaxWarning: 'int' object is not subscriptable; perhaps you missed a comma?
<>:29: SyntaxWarning: 'int' object is not subscriptable; perhaps you missed a comma?
C:\Users\thaku\AppData\Local\Temp\ipykernel_8604\1569203113.py:29: SyntaxWarning: 'int' object is not subscriptable; perhaps you missed a comma?
  df.iloc[1[0]].title


# remove self-recommendation Explicity

In [81]:
%who

ALPHA	 BETA	 collab_scores	 content_scores	 df	 hybrid_recommend	 hybrid_scores	 idx	 indices	 
np	 os	 pd	 pickle	 recommend	 sim_score	 similarity	 user_similarity	 


In [82]:
print(df.head())
print(df.columns)

                              title  \
0  my little pony: a new generation   
1                      the starling   
2                      je suis karl   
3  confessions of an invisible girl   
4                         intrusion   

                                              tags_x  \
0  robert cullen, josé lui ucha vanessa hudgens, ...   
1  theodor melfi melissa mccarthy, chri o'dowd, k...   
2  christian schwochow luna wedler, janni niewöhn...   
3  bruno garotti klara castanho, lucca picon, júl...   
4  adam salki freida pinto, logan marshall-green,...   

                                               genre  popularity  vote_count  \
0  Animation, Family, Fantasy, Comedy, Music, Adv...     137.815       190.0   
1                                              Drama      27.515       174.0   
2                                              Drama      19.294        20.0   
3                                      Comedy, Drama      62.097       144.0   
4                           

In [83]:
indices = pd.Series(
    df.index,
    index=df['title']
).drop_duplicates()

In [84]:
print('my little pony: a new generation' in indices.index)

True


In [85]:
idx = indices['my little pony: a new generation']
print(idx)

0


In [86]:
sim_score = list(enumerate(similarity[idx]))

sim_score = sorted(
    sim_score,
    key=lambda x: x[1],
    reverse=True
)

print(sim_score[:10])

[(0, np.float64(1.0000000000000002)), (489, np.float64(0.1794065731303197)), (1146, np.float64(0.14639038183252748)), (683, np.float64(0.1439116009810935)), (624, np.float64(0.13805228804789463)), (1696, np.float64(0.13545505489240894)), (74, np.float64(0.12100568565140603)), (1587, np.float64(0.11352915000233335)), (50, np.float64(0.11174354633973721)), (1325, np.float64(0.1098235706530666))]


# test

In [87]:
hybrid_recommend("avatar")

hybrid_recommend("titanic")

hybrid_recommend("batman")

'Movie not found'

In [88]:
def recommend(title):
    idx = indices[title]

    sim_score = list(enumerate(similarity[idx]))
    sim_score = sorted(sim_score, key=lambda x: x[1], reverse=True)

    for i in sim_score[1:6]:
        print(df.iloc[i[0]]['title'])



# Evaluation comparison

In [89]:
print("content recommendation:")

print(df.iloc[
    np.argsort(
        similarity[10]
    )[-6:1]
]['title'].values)

print("\nHybrid recommendation:")


content recommendation:
<StringArray>
[]
Length: 0, dtype: str

Hybrid recommendation:


In [90]:
def hybrid_recommend(movie, n=5):

    try:
        movie = movie.lower().strip()
        titles = df['title'].str.lower()

        idx = df[titles == movie].index[0]

    except:
        return "Movie not found"

    distances = list(
        enumerate(hybrid_scores[idx])
    )

    movie_list = sorted(
        distances,
        key=lambda x: x[1],
        reverse=True
    )[1:n+1]

    recommendations = []

    for i in movie_list:
        recommendations.append(
            df.iloc[i[0]]['title']
        )

    return recommendations

# save hybrid model

In [91]:
pickle.dump(

hybrid_scores,

open(

"../models/hybrid.pkl",

"wb"

)

)

# save movie list

In [92]:
import os

print(os.getcwd())

c:\hybrid_recom_project\smart_hybrid_netflix_recommender\Notebook


In [93]:
import os

print(os.path.exists("models"))

False


In [94]:
import pickle

with open("../Models/hybrid.pkl", "wb") as f:
    pickle.dump(hybrid_scores, f)

with open("../Models/user_movie_list.pkl", "wb") as f:
    pickle.dump(df, f)

print("Files saved successfully!")

Files saved successfully!


In [95]:
import os

print(os.path.exists("../Models"))

True


In [96]:
with open("../Models/movies.pkl", "wb") as f:
    pickle.dump(df, f)

with open("../Models/similarity.pkl", "wb") as f:
    pickle.dump(similarity, f)

with open("../Models/hybrid.pkl", "wb") as f:
    pickle.dump(hybrid_scores, f)